# SBND RWM True Bucket: Errors (Step 6 Item 3)

- Total error of every output value: bootstrap, extraction-choice variants (s² retuned per variant), time slices, ripple; cross-segment covariance
- Reads the `true_bucket.ipynb` outputs + `inputs/s2_tune_inputs.npz`; progress in `OUT_DIR/errors_progress.log`
- Writes `OUT_DIR/true_bucket.csv` (values + errors per segment) + `true_bucket_cov.npz`

## 1. Setup

- Set the paths in the Setup cell: `STORE`, `CALIB_CSV`, `OUT_DIR` (same as `true_bucket.ipynb`)

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd
import pickle

import sys
sys.path.insert(0, '../lib')
import paths
from segments import select_chunks, segment_inputs
from ad_response import SEGMENTS, SEG_TITLE, ECHO_SETS, B_PARS, N_GROUPS, bunch_rms, load_ad_term
from true_bucket import (FF_PERIODS, FT_HEAD_FIRST, FT_SAVE_PARS, FT_ERR_GROUPS, reflection_models, ft_context, ft_values, ff_traces, ft_systematics,
                         ft_ripple_closure, ft_bootstrap, ft_uncertainty, ft_covariance, save_ft_covariance, load_npz, load_tails, load_s2_inputs)

# ===== Set these paths =====
STORE = {1: '/path/to/run1_rwm_traces', 2: '/path/to/run2_rwm_traces'}   # stored raw RWM traces: folders of run_<DAQ run>.npz
CALIB_CSV = {1: '/path/to/run1_calibration.csv', 2: '/path/to/run2_calibration.csv'}   # calibration CSVs from SBND_RWM_CALIBRATION
OUT_DIR = '../output'   # output folder, the same in every notebook
paths.set_paths(store=STORE, calib=CALIB_CSV, out=OUT_DIR)

F_TERM = paths.out_file('ad_term.npz')                # Step 2: train-end probes per AD config
F_TAILS = paths.out_file('train_end.npz')             # Step 2: mean trace after the last bucket per AD config
F_WB = paths.out_file('corrected_overlays.npz')       # Step 2: bucket overlays of the corrected traces per segment
F_FITS = paths.out_file('true_bucket_fits.pkl')       # Steps 4-5: H, taps and group fits
F_DENS = paths.out_file('true_bucket_density.npz')    # Step 6: true bucket b per segment
F_CSV = paths.out_file('true_bucket.csv')             # Step 6: values + errors per segment
F_COV = paths.out_file('true_bucket_cov.npz')         # Step 6: cross-segment covariance of the errors
F_LOG = paths.out_file('errors_progress.log')

HW_VAR = {'BL ±10 %': [dict(bl=1.1), dict(bl=0.9)], 'echoes ±20 %': [dict(ec=1.2), dict(ec=0.8)]}   # hardware variants (× nominal)
ECHO_VARIANTS = {'none': (), **ECHO_SETS}   # echo-model variants
N_SLICES = 3                                # time slices per segment

## 2. Inputs

- Segments, corrected overlays, H, taps and group fits as in `true_bucket.ipynb`; s² tune inputs frozen (SBND stacks, MC D_ν)

In [ ]:
DATA = segment_inputs({**select_chunks(1), **select_chunks(2)})
WB = load_npz(F_WB)
with open(F_FITS, 'rb') as f_:
    Z = pickle.load(f_)
CTX, FIT = ft_context(DATA, WB, fitted=Z), Z['FIT']
RMOD = reflection_models(load_ad_term(F_TERM), load_tails(F_TAILS), list(FF_PERIODS))
S2IN = load_s2_inputs(f'{paths.INPUTS}/s2_tune_inputs.npz')
T0 = time.time()
LOG = open(F_LOG, 'w')


def log(m):
    LOG.write(f'[{time.time() - T0:.0f} s] {m}\n')
    LOG.flush()

## 3. Errors

- Ripple → extraction-choice variants (each refits b) → bootstrap

In [ ]:
TRF = {**ff_traces(1), **ff_traces(2)}
RIP = ft_ripple_closure(CTX, FIT)
log('ripple done')
SY = ft_systematics(CTX, DATA, WB, FIT, HW_VAR, ECHO_VARIANTS, N_SLICES, S2IN, traces=TRF, rmod=RMOD, log=log)
del TRF
BOOT = ft_bootstrap(CTX, WB, FIT)
log('bootstrap done')
UNC = ft_uncertainty({k: ft_values(FIT[k]['mix']) for k in SEGMENTS}, SY, BOOT, RIP)
save_ft_covariance(ft_covariance(SY, UNC, SEGMENTS), SEGMENTS, F_COV)
log('done')
LOG.close()
print(f"nominal s² retune {SY['nominal']['s2']:.4f} ns² ({time.time() - T0:.0f} s)")

## 4. Output Table

- One row per segment: shape and RMS with their errors per group (`_tot` = total), s², group weights and fits

In [ ]:
rows = []
for k in SEGMENTS:
    f = FIT[k]
    rows.append(dict(run=k[0], seg=k[1], period=str(CTX['DSF'][k]['blk_period'][0]), head_first=FT_HEAD_FIRST, **UNC[k],
                     s2=f['s2'], rms_ad=f['mix_ad']['rms'], rms_agg2=f['mix']['rms_agg2'], rms_head=bunch_rms(f['fg'][0]['p']),
                     **{f'w_g{m}': CTX['WTG'][k][m] for m in range(N_GROUPS)}, **{f'{n}_g{m}': f['fg'][m]['p'][n] for m in range(N_GROUPS) for n in B_PARS},
                     **{f'chi2ndof_g{m}': f['fg'][m]['chi2'] / f['fg'][m]['ndof'] for m in range(N_GROUPS)}))
paths.write_csv(pd.DataFrame(rows), F_CSV)
pd.DataFrame({SEG_TITLE[k]: {p: f"{UNC[k][p]:.3f} ± {UNC[k][f'{p}_tot']:.3f}" for p in FT_SAVE_PARS} for k in SEGMENTS}).T